### Генерируем данные для обучения 

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.linalg import eigh
from tqdm import tqdm
import random
import csv

In [6]:
def find_band_gaps(freqs, k_slice=None, fmin=None, fmax=None, min_width=1e-6):
    arr = freqs if k_slice is None else freqs[k_slice]

    band_mins = np.min(arr, axis=0)
    band_maxs = np.max(arr, axis=0)

    gaps = []
    for i in range(arr.shape[1] - 1):
        low = band_maxs[i]
        high = band_mins[i + 1]
        width = high - low

        if width > min_width:
            low_clip = max(low, fmin) if fmin is not None else low
            high_clip = min(high, fmax) if fmax is not None else high

            if high_clip - low_clip > min_width:
                gaps.append({
                    "band_left": i,
                    "band_right": i + 1,
                    "f_low": float(low_clip),
                    "f_high": float(high_clip),
                    "width": float(high_clip - low_clip),
                })

    gaps.sort(key=lambda g: g["width"], reverse=True)
    return gaps


def get_band_gap(E = 70e9, nu = 0.3, rho = 2700, h = 0.002, a = 0.1, mR = 0.027, fR_target = 300, M_order=5):
    kR = mR * (2 * np.pi * fR_target)**2 

    D_const = (E * h**3) / (12 * (1 - nu**2))
    S = a**2          
    mass_plate = rho * h * S

    range_M = np.arange(-M_order, M_order + 1)
    GX, GY = np.meshgrid(range_M, range_M)
    GX, GY = GX.flatten() * (2 * np.pi / a), GY.flatten() * (2 * np.pi / a)
    N_waves = len(GX)

    
    pts = 40
    path_k = np.zeros((3 * pts, 2))
    path_k[0:pts, 0] = np.linspace(0, np.pi/a, pts)
    path_k[pts:2*pts, 0] = np.pi/a
    path_k[pts:2*pts, 1] = np.linspace(0, np.pi/a, pts)
    path_k[2*pts:, 0] = np.linspace(np.pi/a, 0, pts)
    path_k[2*pts:, 1] = np.linspace(np.pi/a, 0, pts)

    freqs_lr = []
    freqs_bare = []

    for kx, ky in path_k:
        K_diag = ((kx + GX)**2 + (ky + GY)**2)**2
        K_mat = np.diag(K_diag)
    
        U = np.ones((N_waves, N_waves))
        I = np.eye(N_waves)

        LHS_b = D_const * S * K_mat
        RHS_b = mass_plate * I
        vals_b, _ = eigh(LHS_b, RHS_b)
        freqs_bare.append(np.sqrt(np.maximum(vals_b, 0)) / (2 * np.pi))

        A = np.zeros((N_waves + 1, N_waves + 1))
        A[:N_waves, :N_waves] = D_const * S * K_mat + kR * U
        A[:N_waves, N_waves] = -kR * np.ones(N_waves)
        A[N_waves, :N_waves] = -kR * np.ones(N_waves)
        A[N_waves, N_waves] = kR
        
        B = np.zeros((N_waves + 1, N_waves + 1))
        B[:N_waves, :N_waves] = mass_plate * I
        B[N_waves, N_waves] = mR
        
        vals_lr, _ = eigh(A, B)
        freqs_lr.append(np.sqrt(np.maximum(vals_lr, 0)) / (2 * np.pi))

    freqs_lr = np.array(freqs_lr)
    freqs_bare = np.array(freqs_bare)

    gaps_all = find_band_gaps(freqs_lr, fmin=0, fmax=800)

    return gaps_all

In [25]:
def generate_sample():
    return {
        "E": random.uniform(60e9, 110e9),
        "nu": random.uniform(0.27, 0.33),
        "rho": random.uniform(2600, 2800),
        "h": random.uniform(0.0015, 0.0025),
        "a": random.uniform(0.08, 0.12),
        "mR": random.uniform(0.02, 0.035),
        "fR_target": random.uniform(30, 500)
    }

def generate_data(size=100, name="dataset.csv", sample_generator=generate_sample, save=True):
    rows = []

    for i in tqdm(range(size)):
        row = sample_generator()
        result = get_band_gap(**row)

        if result and len(result) > 0:
            gap = result[0]
            row["f_low"] = gap["f_low"]
            row["f_high"] = gap["f_high"]
        else:
            row["f_low"] = 0
            row["f_high"] = 0


        rows.append(row)

    
    df = pd.DataFrame(rows)
    if save:
        df.to_csv(name, index=False, encoding='utf-8-sig')
        print(f"Данные сохранены в {name}")

    return df

In [26]:
generate_data(size=10, name="train_data.csv", save=False)

100%|██████████| 10/10 [00:02<00:00,  4.58it/s]


,E,nu,rho,h,a,mR,fR_target,f_low,f_high
0,8.276182e+10,0.312362,2739.793422,0.002459,0.095318,0.022939,163.854422,162.024417,191.411452
1,7.929245e+10,0.305148,2699.493916,0.002089,0.105254,0.027160,495.890337,392.548098,486.588157
2,6.756007e+10,0.318930,2623.364134,0.001839,0.110007,0.029650,342.178042,273.883688,369.028015
3,1.013798e+11,0.287501,2789.871219,0.001779,0.098980,0.033368,168.027705,161.470671,215.290516
4,6.587063e+10,0.277387,2769.049859,0.002344,0.108739,0.020962,280.921679,264.749476,310.831497
5,7.204034e+10,0.292919,2759.196601,0.001801,0.111964,0.021801,359.554140,293.509364,348.150982
6,9.548498e+10,0.306795,2636.737888,0.002143,0.089142,0.032443,108.949557,108.077276,142.613204
7,7.194396e+10,0.275586,2695.005177,0.002453,0.091410,0.032383,303.774838,286.617322,375.179331
8,8.001039e+10,0.300162,2709.387052,0.002054,0.102483,0.022334,358.048067,321.857553,405.760087
9,1.095642e+11,0.305290,2785.119294,0.002354,0.080450,0.024180,203.996605,201.711448,254.649296


### Теперь сгенерируем данные для обучения

In [ ]:
generate_data(size=2000, name="/Users/grigorijevgenev/Desktop/CourseWorkPINN/data/generated_data/train_data.csv")
generate_data(size=200, name="/Users/grigorijevgenev/Desktop/CourseWorkPINN/data/generated_data/val_data.csv")
generate_data(size=200, name="/Users/grigorijevgenev/Desktop/CourseWorkPINN/data/generated_data/test_data.csv")

  1%|▏         | 26/2000 [00:05<07:03,  4.66it/s]